# 09 — Robustness Check: Second Algorithm Family

**Purpose.** Repeats notebook 08's exact three-step comparison (baseline
momentum-only -> + rate level -> + rate-of-change/interaction) with a
second algorithm family, gradient-boosted trees (`xgboost`, already in
`requirements.txt`), and checks whether the same ranking of marginal
contributions holds. Per `docs/phase0_framing.md`'s Success Criteria table,
this robustness check ("same ranking of contributions across >= 2 algorithm
families") is what confirms a finding isn't specific to one algorithm.

**Why XGBoost and not Random Forest.** Both are listed as acceptable
options; `xgboost` is what this project's `requirements.txt` already pins
(`xgboost==3.4.1`), so no new dependency is introduced. This is a judgment
call, not something `docs/phase0_framing.md` itself specifies.

**Reuses notebook 08's locked baseline and CV folds.** The baseline is
locked exactly once (`outputs/baseline_lock.json`) across the whole project,
not per algorithm -- this notebook loads that lock file and the same
`outputs/cv_folds.json` fold boundaries rather than relocking or
re-deriving either. If the lock file is missing, this notebook stops and
says to run notebook 08 first.

Logistic regression's interaction-coefficient sign check (notebook 08,
Section 9) has no equivalent here: XGBoost doesn't expose a single scalar
coefficient the same way a linear model does, so that check is not
attempted a second time in this notebook.


## 1. Setup

In [9]:
import json
import os

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score

from housing_risk import features

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Load the feature matrix, CV folds, and the locked baseline

The locked baseline (`outputs/baseline_lock.json`) is specific to the
logistic-regression algorithm from notebook 08 -- its `pooled_roc_auc` is
still the reference number quoted in `docs/phase0_framing.md`'s Success
Criteria table (the baseline is locked once, project-wide, not
per-algorithm), but this notebook's own step deltas are computed against
its *own* XGBoost baseline run below, since a fair robustness comparison
needs the same algorithm on both sides of each delta.


In [10]:
FEATURES_PATH = "../data/processed/train_features.csv"
FOLDS_PATH = "../outputs/cv_folds.json"
BASELINE_LOCK_PATH = "../outputs/baseline_lock.json"

assert os.path.exists(FEATURES_PATH), (
    f"{FEATURES_PATH} not found -- run notebooks/07_feature_engineering.ipynb first."
)
assert os.path.exists(FOLDS_PATH), (
    f"{FOLDS_PATH} not found -- run notebooks/05_split_cv.ipynb first."
)
assert os.path.exists(BASELINE_LOCK_PATH), (
    f"{BASELINE_LOCK_PATH} not found -- run notebooks/08_modeling_sequence.ipynb first; "
    "this notebook reuses its locked baseline rather than relocking one here."
)

matrix = pd.read_csv(FEATURES_PATH, parse_dates=["month"])
with open(FOLDS_PATH) as fh:
    folds = json.load(fh)
for fold in folds:
    for key in ("train_start", "train_end", "val_start", "val_end"):
        fold[key] = pd.Timestamp(fold[key])
with open(BASELINE_LOCK_PATH) as fh:
    logreg_baseline_lock = json.load(fh)

print(f"Loaded {len(matrix):,} feature-matrix rows")
print(f"Loaded {len(folds)} CV folds")
print(f"Loaded locked logistic-regression baseline: pooled ROC-AUC = {logreg_baseline_lock['pooled_roc_auc']:.4f}")


Loaded 164,460 feature-matrix rows
Loaded 11 CV folds
Loaded locked logistic-regression baseline: pooled ROC-AUC = 1.0000


## 3. Feature steps (identical to notebook 08)

In [11]:
MOMENTUM_FEATURES = ["momentum_1m", "momentum_3m", "momentum_6m"]
RATE_LEVEL_FEATURES = ["mortgage_rate_level"]
RATE_CHANGE_FEATURES = ["mortgage_rate_roc", "rate_roc_x_momentum"]

FEATURE_STEPS = {
    "baseline": MOMENTUM_FEATURES,
    "step2_rate_level": MOMENTUM_FEATURES + RATE_LEVEL_FEATURES,
    "step3_rate_change": MOMENTUM_FEATURES + RATE_LEVEL_FEATURES + RATE_CHANGE_FEATURES,
}
assert FEATURE_STEPS["step3_rate_change"] == features.FEATURE_COLUMNS, (
    "STOP: this notebook's step-3 feature list has drifted from housing_risk.features.FEATURE_COLUMNS"
)
TARGET = "decline_label"


## 4. Minimum positives per CV fold (identical gate and threshold to notebook 08)

In [12]:
MIN_FOLD_POSITIVES = 20  # identical to notebook 08, for a like-for-like comparison

def fold_val_mask(fold):
    return (matrix["month"] >= fold["val_start"]) & (matrix["month"] <= fold["val_end"])

fold_status = []
for fold in folds:
    val_rows = matrix.loc[fold_val_mask(fold)]
    n_positive = int(val_rows[TARGET].sum())
    status = "ok" if n_positive >= MIN_FOLD_POSITIVES else "skip"
    fold_status.append({**fold, "n_val_rows": len(val_rows), "n_val_positives": n_positive, "status": status})

ok_folds = [f for f in fold_status if f["status"] == "ok"]
skipped_folds = [f for f in fold_status if f["status"] == "skip"]
print(f"{len(ok_folds)} of {len(folds)} folds usable (>= {MIN_FOLD_POSITIVES} validation positives)")
if skipped_folds:
    print(f"Skipped folds: {[f['fold_id'] for f in skipped_folds]}")
assert len(ok_folds) >= 3, "STOP: fewer than 3 usable CV folds after the minimum-positives gate."


11 of 11 folds usable (>= 20 validation positives)


## 5. Cross-validated scoring, per feature step (XGBoost)

Same expanding-window folds, same feature steps, same OOF-pooling approach
as notebook 08 -- only the model changes. XGBoost's native handling of
feature scale means no `StandardScaler` step is needed here (unlike the
logistic-regression pipeline in notebook 08).


In [13]:
def make_model():
    return xgb.XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

def cv_score_step(feature_cols, ok_folds=ok_folds):
    per_fold_rows = []
    oof_rows = []
    for fold in ok_folds:
        train_mask = matrix["month"] <= fold["train_end"]
        val_mask = fold_val_mask(fold)
        train_df = matrix.loc[train_mask]
        val_df = matrix.loc[val_mask]

        model = make_model()
        model.fit(train_df[feature_cols], train_df[TARGET])
        val_pred = model.predict_proba(val_df[feature_cols])[:, 1]

        fold_roc_auc = roc_auc_score(val_df[TARGET], val_pred)
        fold_pr_auc = average_precision_score(val_df[TARGET], val_pred)
        fold_prevalence = float(val_df[TARGET].mean())
        per_fold_rows.append({
            "fold_id": fold["fold_id"], "roc_auc": fold_roc_auc, "pr_auc": fold_pr_auc,
            "prevalence": fold_prevalence, "n_val": len(val_df),
        })
        oof_rows.append(pd.DataFrame({
            "fold_id": fold["fold_id"], "month": val_df["month"].to_numpy(),
            "y_true": val_df[TARGET].to_numpy(), "y_pred": val_pred,
        }))

    per_fold = pd.DataFrame(per_fold_rows)
    oof = pd.concat(oof_rows, ignore_index=True)
    pooled_roc_auc = roc_auc_score(oof["y_true"], oof["y_pred"])
    pooled_pr_auc = average_precision_score(oof["y_true"], oof["y_pred"])
    summary = {
        "mean_roc_auc": float(per_fold["roc_auc"].mean()),
        "std_roc_auc": float(per_fold["roc_auc"].std()),
        "mean_pr_auc": float(per_fold["pr_auc"].mean()),
        "std_pr_auc": float(per_fold["pr_auc"].std()),
        "pooled_roc_auc": float(pooled_roc_auc),
        "pooled_pr_auc": float(pooled_pr_auc),
        "pooled_prevalence": float(oof["y_true"].mean()),
        "n_folds": len(ok_folds),
    }
    return summary, per_fold, oof

step_summaries = {}
step_oof = {}
for step_name, cols in FEATURE_STEPS.items():
    summary, per_fold, oof = cv_score_step(cols)
    step_summaries[step_name] = summary
    step_oof[step_name] = oof
    print(f"{step_name}: mean ROC-AUC={summary['mean_roc_auc']:.4f} (+/-{summary['std_roc_auc']:.4f}), "
          f"pooled ROC-AUC={summary['pooled_roc_auc']:.4f}, pooled PR-AUC={summary['pooled_pr_auc']:.4f}")

print(f"\nFor reference, the locked logistic-regression baseline's pooled ROC-AUC was "
      f"{logreg_baseline_lock['pooled_roc_auc']:.4f}; this XGBoost baseline's own pooled ROC-AUC is "
      f"{step_summaries['baseline']['pooled_roc_auc']:.4f} -- these are two different algorithms' "
      "baseline scores, not expected to match, and are not compared to each other directly below.")


baseline: mean ROC-AUC=0.8787 (+/-0.0284), pooled ROC-AUC=0.8979, pooled PR-AUC=0.4070
step2_rate_level: mean ROC-AUC=0.8691 (+/-0.0284), pooled ROC-AUC=0.8895, pooled PR-AUC=0.3395
step3_rate_change: mean ROC-AUC=0.8691 (+/-0.0404), pooled ROC-AUC=0.8918, pooled PR-AUC=0.3463

For reference, the locked logistic-regression baseline's pooled ROC-AUC was 1.0000; this XGBoost baseline's own pooled ROC-AUC is 0.8979 -- these are two different algorithms' baseline scores, not expected to match, and are not compared to each other directly below.


## 6. Quarter-clustered bootstrap on marginal AUC deltas (identical method to notebook 08)

In [14]:
N_BOOTSTRAP = 2000  # identical to notebook 08
rng = np.random.default_rng(RANDOM_STATE)

def quarter_clustered_bootstrap_delta(oof_a, oof_b, n_bootstrap=N_BOOTSTRAP, rng=rng):
    assert oof_a["month"].equals(oof_b["month"]), "oof_a/oof_b must be row-aligned on the same validation rows"
    assert (oof_a["y_true"].to_numpy() == oof_b["y_true"].to_numpy()).all(), "y_true must match row-for-row"

    df = pd.DataFrame({
        "month": oof_a["month"].to_numpy(),
        "y_true": oof_a["y_true"].to_numpy(),
        "pred_a": oof_a["y_pred"].to_numpy(),
        "pred_b": oof_b["y_pred"].to_numpy(),
    })
    quarters = df["month"].unique()
    by_quarter = {q: df.loc[df["month"] == q] for q in quarters}

    deltas = []
    for _ in range(n_bootstrap):
        draw = rng.choice(quarters, size=len(quarters), replace=True)
        resampled = pd.concat([by_quarter[q] for q in draw], ignore_index=True)
        if resampled["y_true"].nunique() < 2:
            continue
        auc_a = roc_auc_score(resampled["y_true"], resampled["pred_a"])
        auc_b = roc_auc_score(resampled["y_true"], resampled["pred_b"])
        deltas.append(auc_b - auc_a)

    deltas = np.array(deltas)
    point_delta = roc_auc_score(df["y_true"], df["pred_b"]) - roc_auc_score(df["y_true"], df["pred_a"])
    ci_low, ci_high = float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))
    significant = not (ci_low <= 0.0 <= ci_high)
    return {
        "point_delta": float(point_delta), "ci_95_low": ci_low, "ci_95_high": ci_high,
        "n_bootstrap_used": int(len(deltas)), "significant": bool(significant),
    }

delta_2_vs_1 = quarter_clustered_bootstrap_delta(step_oof["baseline"], step_oof["step2_rate_level"])
delta_3_vs_2 = quarter_clustered_bootstrap_delta(step_oof["step2_rate_level"], step_oof["step3_rate_change"])

print("Step 2 (+ rate level) vs. baseline, ROC-AUC delta:")
print(f"  point estimate: {delta_2_vs_1['point_delta']:+.4f}  "
      f"95% CI: [{delta_2_vs_1['ci_95_low']:+.4f}, {delta_2_vs_1['ci_95_high']:+.4f}]")
print("Step 3 (+ rate-of-change/interaction) vs. step 2, ROC-AUC delta:")
print(f"  point estimate: {delta_3_vs_2['point_delta']:+.4f}  "
      f"95% CI: [{delta_3_vs_2['ci_95_low']:+.4f}, {delta_3_vs_2['ci_95_high']:+.4f}]")


Step 2 (+ rate level) vs. baseline, ROC-AUC delta:
  point estimate: -0.0084  95% CI: [-0.0149, -0.0017]
Step 3 (+ rate-of-change/interaction) vs. step 2, ROC-AUC delta:
  point estimate: +0.0023  95% CI: [-0.0042, +0.0088]


## 7. Robustness check: does the same ranking hold?

Per `docs/phase0_framing.md`'s Success Criteria table, the robustness row
asks whether the **same ranking of contributions** holds across algorithm
families, not whether the exact AUC numbers match -- logistic regression and
XGBoost are different models and are expected to produce different absolute
scores.


In [15]:
LOGREG_RESULTS_PATH = "../outputs/modeling_sequence_logreg.json"
assert os.path.exists(LOGREG_RESULTS_PATH), (
    f"{LOGREG_RESULTS_PATH} not found -- run notebooks/08_modeling_sequence.ipynb first; "
    "this check needs its persisted results to compare against."
)
with open(LOGREG_RESULTS_PATH) as fh:
    logreg_results = json.load(fh)

def is_positive_contribution(delta_result, threshold=0.01):
    return delta_result["significant"] and delta_result["point_delta"] >= threshold

logreg_step2_positive = is_positive_contribution(logreg_results["delta_step2_vs_baseline"])
logreg_step3_positive = is_positive_contribution(logreg_results["delta_step3_vs_step2"])
xgb_step2_positive = is_positive_contribution(delta_2_vs_1)
xgb_step3_positive = is_positive_contribution(delta_3_vs_2)

print(f"Logistic regression -- step 2 meaningful lift: {logreg_step2_positive}, "
      f"step 3 meaningful lift: {logreg_step3_positive}")
print(f"XGBoost             -- step 2 meaningful lift: {xgb_step2_positive}, "
      f"step 3 meaningful lift: {xgb_step3_positive}")

ranking_matches = (logreg_step2_positive == xgb_step2_positive) and (logreg_step3_positive == xgb_step3_positive)
print(f"\nSame ranking of contributions across both algorithm families: {ranking_matches}")
if ranking_matches:
    print("Robustness check PASSED (per docs/phase0_framing.md's Success Criteria table).")
else:
    print(
        "Robustness check DID NOT hold: the two algorithm families disagree on which steps add a "
        "significant, threshold-clearing lift. This is a genuine, reportable finding -- it means "
        "the conclusion is sensitive to model choice, not a failure of this notebook."
    )


Logistic regression -- step 2 meaningful lift: False, step 3 meaningful lift: False
XGBoost             -- step 2 meaningful lift: False, step 3 meaningful lift: False

Same ranking of contributions across both algorithm families: True
Robustness check PASSED (per docs/phase0_framing.md's Success Criteria table).


## 8. Persist results

In [16]:
results = {
    "algorithm": "xgboost",
    "feature_steps": FEATURE_STEPS,
    "step_summaries": step_summaries,
    "min_fold_positives": MIN_FOLD_POSITIVES,
    "n_folds_used": len(ok_folds),
    "delta_step2_vs_baseline": delta_2_vs_1,
    "delta_step3_vs_step2": delta_3_vs_2,
    "n_bootstrap": N_BOOTSTRAP,
    "random_state": RANDOM_STATE,
    "ranking_matches_logreg": bool(ranking_matches),
}
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/modeling_sequence_xgboost.json", "w") as fh:
    json.dump(results, fh, indent=2)
print("Saved outputs/modeling_sequence_xgboost.json")


Saved outputs/modeling_sequence_xgboost.json


## 9. Findings

- **Reused notebook 08's locked baseline and CV folds** rather than
  relocking or re-deriving either -- this notebook's own baseline run is a
  fresh XGBoost fit on the same folds, not a second lock.
- **Same feature steps, same minimum-positives gate, same quarter-clustered
  bootstrap method** as notebook 08, so the two algorithms' results are
  comparable on ranking even though their absolute AUC numbers are not
  expected to match.
- **The robustness check's result is reported explicitly** (Section 7): a
  matching ranking is a pass; a mismatched ranking is reported as a genuine
  finding (the conclusion is sensitive to model choice) rather than treated
  as a notebook failure.
- **No interaction-coefficient sign check here** -- XGBoost doesn't expose
  a single scalar coefficient the way logistic regression does, so that
  check (notebook 08, Section 9) is not repeated in this notebook.
- **Not done here:** the held-out test set is never touched in this
  notebook either. Final scoring against it, for both algorithms, is
  notebook 10's job.
